#### **Parameters**

In [21]:
# # =============================================================================
# # ORIGAMI CLAIMS - EXECUTION PARAMETERS (Fabric-friendly)
# # Edit these values or override via pipeline / notebook run
# # =============================================================================

# # Parameter 1: Load Type ('INCREMENTAL' or 'FULL')
# PARAM_LOAD_TYPE = "INCREMENTAL"

# # Parameter 2: Line of Business Selection ('CL', 'PL', or 'BOTH')
# PARAM_LOB = "BOTH"

# # Parameter 3: Run Date ('YYYY-MM-DD' or empty string/None for Today)
# PARAM_RUN_DATE = "2026-11-1"

StatementMeta(, 9c861cff-517b-43c1-b78c-c482bdfba29a, 23, Finished, Available, Finished, False)

#### **Imports, fixed settings, parameter checks**

In [22]:
import json
from datetime import datetime
from decimal import Decimal
from pyspark.sql import functions as F

# ---- Fixed settings (not parameters) ----
RAWCLM           = "AnalyticsData.AnalyticsDB.RAWCLM"
DM               = "AnalyticsData.AnalyticsDB.DM"
BASE_OUTPUT_PATH = "Files"
CL_CODES = "9379,9380,86785,9382,9381,11676,14485,11685,11686"
PL_CODES = "11677,11678,11679,11680,11681,9383,11682,9358,11683,11684,11687,11688"

# ---- Load type ----
load_type = (PARAM_LOAD_TYPE or "").strip().upper()
if load_type not in ("INCREMENTAL", "FULL"):
    raise ValueError(f"Invalid PARAM_LOAD_TYPE: {PARAM_LOAD_TYPE}")

# ---- LOB ----
lob_param = (PARAM_LOB or "").strip().upper()
if lob_param not in ("CL", "PL", "BOTH"):
    raise ValueError(f"Invalid PARAM_LOB: {PARAM_LOB}")

# ---- Run date (empty / None = today) ----
if PARAM_RUN_DATE and str(PARAM_RUN_DATE).strip():
    run_date = datetime.strptime(str(PARAM_RUN_DATE).strip(), "%Y-%m-%d").date()
else:
    run_date = datetime.now().date()

print(f"Load Type : {load_type}")
print(f"LOB       : {lob_param}")
print(f"Run Date  : {run_date}")

StatementMeta(, 9c861cff-517b-43c1-b78c-c482bdfba29a, 24, Finished, Available, Finished, False)

Load Type : INCREMENTAL
LOB       : BOTH
Run Date  : 2026-11-01


#### **Helper functions**

In [23]:
def clean_text_columns(df):
    """Remove line breaks and trim spaces in every text column."""
    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(
                col_name,
                F.trim(F.regexp_replace(F.col(col_name), r"[\r\n]+", " "))
            )
    return df


def write_origami_csv(df, lob_prefix, run_date, base_path=BASE_OUTPUT_PATH):
    """
    Writes one CSV: Files/<PA|CA>/<MM-DD-YYYY>/<PA|CA>_<MMDDYYYY>.csv
    Quotes inside text are written as "" (standard CSV), not \"
    """
    folder_date = run_date.strftime("%m-%d-%Y")
    file_date   = run_date.strftime("%m%d%Y")

    target_dir = f"{base_path}/{lob_prefix}/{folder_date}"
    temp_dir   = f"{target_dir}/temp_{lob_prefix}_{file_date}"
    final_file = f"{target_dir}/{lob_prefix}_{file_date}.csv"

    df = clean_text_columns(df)

    (df.coalesce(1)
       .write.mode("overwrite")
       .option("header", "false")
       .option("quote", '"')
       .option("escape", '"')          
       .csv(temp_dir))

    part_file = [f.path for f in notebookutils.fs.ls(temp_dir)
                 if f.name.startswith("part-") and f.name.endswith(".csv")][0]

    if notebookutils.fs.exists(final_file):
        notebookutils.fs.rm(final_file, True)
    notebookutils.fs.mv(part_file, final_file)
    notebookutils.fs.rm(temp_dir, True)

    print(f"File written: {final_file}")
    return final_file


STATUS_CASE = """
CASE
    WHEN st.CODE_DESC IN ('Closed', 'Closed No Coverage', 'Closed with Salvage') THEN 'Closed'
    WHEN st.CODE_DESC = 'Closed Peril Not Covered' THEN 'Peril Not Covered'
    WHEN st.CODE_DESC = 'Closed with Subro'        THEN 'Subrogation'
    WHEN st.CODE_DESC = 'Closed under Deductible'  THEN 'Under Deductible'
    WHEN st.CODE_DESC = 'Closed withdrawn'         THEN 'Withdrawn'
    WHEN st.CODE_DESC IN ('New Claim', 'Open', 'Open with Coverage',
                          'Open with Reinsurance', 'Reopened') THEN 'Open'
    ELSE st.CODE_DESC
END
"""

StatementMeta(, 9c861cff-517b-43c1-b78c-c482bdfba29a, 25, Finished, Available, Finished, False)

#### **Claim base**

In [24]:
date_filter = f"AND cd.LAST_UPDATE_DATE = DATE'{run_date}'" if load_type == "INCREMENTAL" else ""

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_claim_base AS
WITH claim_dt AS (
    SELECT c.*,
           COALESCE(
               to_date(to_timestamp(c.DTTM_RCD_LAST_UPD, 'yyyyMMddHHmmss')),
               to_date(substring(c.DTTM_RCD_LAST_UPD, 1, 8), 'yyyyMMdd')
           ) AS LAST_UPDATE_DATE
    FROM {RAWCLM}.CLAIM c
)
SELECT
    cd.CLAIM_ID,
    cd.CLAIM_NUMBER,
    cd.EVENT_ID,
    cd.DATE_OF_CLAIM,
    cd.LOSS_DESCRIPTION,
    p.POLICY_NUMBER,
    p.POLICY_LOB_CODE,
    CASE WHEN p.POLICY_LOB_CODE IN ({CL_CODES}) THEN 'CL'
         WHEN p.POLICY_LOB_CODE IN ({PL_CODES}) THEN 'PL'
    END AS LOB_GROUP,
    {STATUS_CASE} AS CLAIM_STATUS
FROM claim_dt cd
JOIN {RAWCLM}.CLAIM_X_POLICY cxp ON cd.CLAIM_ID   = cxp.CLAIM_ID
JOIN {RAWCLM}.POLICY p           ON cxp.POLICY_ID = p.POLICY_ID
LEFT JOIN {RAWCLM}.CODES_TEXT st ON cd.CLAIM_STATUS_CODE = st.CODE_ID
WHERE p.POLICY_LOB_CODE IN ({CL_CODES},{PL_CODES})
{date_filter}
""")

print("Claims selected:", spark.table("vw_claim_base").count())

StatementMeta(, 9c861cff-517b-43c1-b78c-c482bdfba29a, 26, Finished, Available, Finished, False)

Claims selected: 0


#### **Financials and earned premium**

In [25]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_claim_fin AS
SELECT
    CLAIM_ID,
    SUM(INCURRED_AMOUNT) AS INCURRED,
    SUM(PAID_TOTAL)      AS PAID,
    SUM(RESERVE_AMOUNT)  AS RESERVES,
    SUM(CASE WHEN RESERVE_TYPE_CODE = 7822 THEN PAID_TOTAL ELSE 0 END) AS EXPENSES
FROM {RAWCLM}.RESERVE_CURRENT
GROUP BY CLAIM_ID
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_policy_premium AS
SELECT PolicyNumber, SUM(`PremiumEarned(Atr)`) AS EARNED_PREMIUM
FROM {DM}.FactPolicyMonthlyDataStore
GROUP BY PolicyNumber
""")

StatementMeta(, 9c861cff-517b-43c1-b78c-c482bdfba29a, 27, Finished, Available, Finished, False)

DataFrame[]

#### **CL extract (13 columns)**

In [26]:
df_cl_final = spark.sql("""
SELECT
    substring(cb.DATE_OF_CLAIM, 1, 8)                        AS `Loss Date`,
    cb.LOSS_DESCRIPTION                                      AS `Loss Description`,
    cb.CLAIM_NUMBER                                          AS `Claim Number`,
    cb.POLICY_NUMBER                                         AS `Policy Number`,
    cb.CLAIM_STATUS                                          AS `Claim Status`,
    CAST(COALESCE(f.INCURRED, 0) AS DECIMAL(20,2))           AS `Incurred`,
    CAST(COALESCE(f.PAID, 0)     AS DECIMAL(20,2))           AS `Paid`,
    CAST(COALESCE(f.RESERVES, 0) AS DECIMAL(20,2))           AS `Reserves`,
    CAST(COALESCE(f.EXPENSES, 0) AS DECIMAL(20,2))           AS `Expenses`,
    CAST(COALESCE(pr.EARNED_PREMIUM, 0) AS DECIMAL(20,2))    AS `Total Earned Premium`,
    CAST(COALESCE(f.INCURRED, 0) AS DECIMAL(20,2))           AS `Total Claims Amount`,
    CAST(CASE WHEN COALESCE(pr.EARNED_PREMIUM, 0) = 0 THEN NULL
              ELSE ROUND((COALESCE(f.INCURRED,0) + COALESCE(f.EXPENSES,0)) / pr.EARNED_PREMIUM * 100, 2)
         END AS DECIMAL(20,2))                               AS `Loss Ratio`,
    CAST(CASE WHEN COALESCE(pr.EARNED_PREMIUM, 0) = 0 THEN NULL
              ELSE ROUND(COALESCE(f.EXPENSES,0) / pr.EARNED_PREMIUM * 100, 2)
         END AS DECIMAL(20,2))                               AS `Expense Ratio`
FROM vw_claim_base cb
LEFT JOIN vw_claim_fin f       ON cb.CLAIM_ID      = f.CLAIM_ID
LEFT JOIN vw_policy_premium pr ON cb.POLICY_NUMBER = pr.PolicyNumber
WHERE cb.LOB_GROUP = 'CL'
""")

print("CL rows:", df_cl_final.count())

StatementMeta(, 9c861cff-517b-43c1-b78c-c482bdfba29a, 28, Finished, Available, Finished, False)

CL rows: 0


#### **PL extract**

In [27]:
df_pl_final = spark.sql(f"""
WITH pl_claims AS (
    SELECT * FROM vw_claim_base WHERE LOB_GROUP = 'PL'
),
vehicle_driver AS (
    SELECT
        vx.CLAIM_ID, vx.UNIT_ID,
        v.VEHICLE_YEAR, v.VEHICLE_MAKE, v.VEHICLE_MODEL,
        e.FIRST_NAME, e.MIDDLE_NAME, e.LAST_NAME,
        ROW_NUMBER() OVER (PARTITION BY vx.CLAIM_ID, vx.UNIT_ID
                           ORDER BY d.DRIVER_EID ASC) AS rn
    FROM {RAWCLM}.VEHICLE_X_ACC_DATE vx
    JOIN {RAWCLM}.VEHICLE v      ON vx.UNIT_ID   = v.UNIT_ID
    LEFT JOIN {RAWCLM}.DRIVER d  ON v.DRIVER_EID = d.DRIVER_EID
    LEFT JOIN {RAWCLM}.ENTITY e  ON d.DRIVER_EID = e.ENTITY_ID
    WHERE vx.CLAIM_ID IN (SELECT CLAIM_ID FROM pl_claims)
),
not_insured AS (
    SELECT
        ux.CLAIM_ID, ux.UNIT_ID,
        MAX(CASE WHEN uxs.NON_INSRD_FLAG IN (-1, 1) THEN 1 ELSE 0 END) AS NOT_INSURED
    FROM {RAWCLM}.UNIT_X_CLAIM ux
    JOIN {RAWCLM}.UNIT_X_CLAIM_SUPP uxs ON ux.UNIT_ROW_ID = uxs.UNIT_ROW_ID
    GROUP BY ux.CLAIM_ID, ux.UNIT_ID
),
exposure AS (
    SELECT
        rc.CLAIM_ID, pu.UNIT_ID,
        ct.CODE_DESC       AS EXPOSURE_TYPE,
        SUM(rc.PAID_TOTAL) AS EXPOSURE_AMOUNT
    FROM {RAWCLM}.CLAIMANT cl
    JOIN {RAWCLM}.RESERVE_CURRENT rc
         ON cl.CLAIM_ID = rc.CLAIM_ID AND cl.CLAIMANT_EID = rc.CLAIMANT_EID
    LEFT JOIN {RAWCLM}.COVERAGE_X_LOSS cxl   ON rc.POLCVG_LOSS_ROW_ID  = cxl.CVG_LOSS_ROW_ID
    LEFT JOIN {RAWCLM}.POLICY_X_CVG_TYPE pct ON cxl.POLCVG_ROW_ID      = pct.POLCVG_ROW_ID
    LEFT JOIN {RAWCLM}.POLICY_X_UNIT pu      ON pct.POLICY_UNIT_ROW_ID = pu.POLICY_UNIT_ROW_ID
    LEFT JOIN {RAWCLM}.CODES_TEXT ct         ON pct.COVERAGE_TYPE_CODE = ct.CODE_ID
    GROUP BY rc.CLAIM_ID, pu.UNIT_ID, ct.CODE_DESC
),

-- CHANGED: Liability Percentage = CLAIM_SUPP.LIAB_PCT_TEXT (CLAIM joined to CLAIM_SUPP on CLAIM_ID)
claim_liab AS (
    SELECT a.CLAIM_ID, MAX(b.LIAB_PCT_TEXT) AS LIAB_PCT_TEXT
    FROM {RAWCLM}.CLAIM a
    JOIN {RAWCLM}.CLAIM_SUPP b ON a.CLAIM_ID = b.CLAIM_ID
    GROUP BY a.CLAIM_ID
),

policyholder AS (
    SELECT
        pi.EVENT_ID,
        concat_ws(' ', e.FIRST_NAME, e.LAST_NAME) AS POLICYHOLDER_NAME,
        ROW_NUMBER() OVER (PARTITION BY pi.EVENT_ID ORDER BY pi.PI_EID) AS rn
    FROM {RAWCLM}.PERSON_INVOLVED pi
    JOIN {RAWCLM}.ENTITY e ON pi.PI_EID = e.ENTITY_ID
    WHERE pi.PI_TYPE_CODE = 6910
)

SELECT
    substring(cb.DATE_OF_CLAIM, 1, 8)                        AS `Loss Date`,
    vd.FIRST_NAME                                            AS `Driver First Name`,
    vd.MIDDLE_NAME                                           AS `Driver Middle Name`,
    vd.LAST_NAME                                             AS `Driver Last Name`,
    CASE WHEN ni.NOT_INSURED = 1 THEN 'Yes' ELSE 'No' END    AS `Driver Not Insured Flag`,
    CAST(vd.VEHICLE_YEAR AS STRING)                          AS `Vehicle Year`,
    vd.VEHICLE_MAKE                                          AS `Vehicle Make`,
    vd.VEHICLE_MODEL                                         AS `Vehicle Model`,
    CAST(COALESCE(ex.EXPOSURE_AMOUNT, 0) AS DECIMAL(20,2))   AS `Exposure Amount`,
    ex.EXPOSURE_TYPE                                         AS `Exposure Type`,
    lb.LIAB_PCT_TEXT                                         AS `Liability Percentage`,  
    CAST(NULL AS STRING)                                     AS `Bodily Injury or Death`,
    cb.LOSS_DESCRIPTION                                      AS `Loss Description`,
    ph.POLICYHOLDER_NAME                                     AS `Policyholder Name`,
    cb.CLAIM_NUMBER                                          AS `Claim Number`,
    cb.POLICY_NUMBER                                         AS `Policy Number`,
    cb.CLAIM_STATUS                                          AS `Claim Status`,
    date_format(current_date(), 'yyyyMMdd')                  AS `Current Date`,
    CAST(COALESCE(pr.EARNED_PREMIUM, 0) AS DECIMAL(20,2))    AS `Total Earned Premium`,
    CAST(COALESCE(f.INCURRED, 0) AS DECIMAL(20,2))           AS `Total Claims Amount`,
    CAST(COALESCE(f.EXPENSES, 0) AS DECIMAL(20,2))           AS `Expenses`,
    CAST(CASE WHEN COALESCE(pr.EARNED_PREMIUM, 0) = 0 THEN NULL
              ELSE ROUND((COALESCE(f.INCURRED,0) + COALESCE(f.EXPENSES,0)) / pr.EARNED_PREMIUM * 100, 2)
         END AS DECIMAL(20,2))                               AS `Loss Ratio`,
    CAST(CASE WHEN COALESCE(pr.EARNED_PREMIUM, 0) = 0 THEN NULL
              ELSE ROUND(COALESCE(f.EXPENSES,0) / pr.EARNED_PREMIUM * 100, 2)
         END AS DECIMAL(20,2))                               AS `Expense Ratio`
FROM pl_claims cb
LEFT JOIN vehicle_driver vd    ON cb.CLAIM_ID = vd.CLAIM_ID AND vd.rn = 1
LEFT JOIN not_insured ni       ON vd.CLAIM_ID = ni.CLAIM_ID AND vd.UNIT_ID = ni.UNIT_ID
LEFT JOIN exposure ex          ON vd.CLAIM_ID = ex.CLAIM_ID AND vd.UNIT_ID = ex.UNIT_ID
LEFT JOIN claim_liab lb        ON cb.CLAIM_ID = lb.CLAIM_ID
LEFT JOIN policyholder ph      ON cb.EVENT_ID = ph.EVENT_ID AND ph.rn = 1
LEFT JOIN vw_claim_fin f       ON cb.CLAIM_ID = f.CLAIM_ID
LEFT JOIN vw_policy_premium pr ON cb.POLICY_NUMBER = pr.PolicyNumber
""")

print("PL rows:", df_pl_final.count())

StatementMeta(, 9c861cff-517b-43c1-b78c-c482bdfba29a, 29, Finished, Available, Finished, False)

PL rows: 0


#### **Write files to the lakehouse**

In [28]:
written_files = {}   # {"CL": path, "PL": path}

# Only write and later SFTP-upload files when there is actual data

# CL file
if lob_param in ("CL", "BOTH"):
    if not df_cl_final.rdd.isEmpty():
        written_files["CL"] = write_origami_csv(df_cl_final, "CA", run_date)
    else:
        print("No CL rows for this run_date / selection -> no CL file written.")

# PL file
if lob_param in ("PL", "BOTH"):
    if not df_pl_final.rdd.isEmpty():
        written_files["PL"] = write_origami_csv(df_pl_final, "PA", run_date)
    else:
        print("No PL rows for this run_date / selection -> no PL file written.")

print("Files written:", written_files)

StatementMeta(, 9c861cff-517b-43c1-b78c-c482bdfba29a, 30, Finished, Available, Finished, False)

No CL rows for this run_date / selection -> no CL file written.
No PL rows for this run_date / selection -> no PL file written.
Files written: {}


#### **Write files to SFTP**

In [29]:
# =============================================================================
# SFTP UPLOAD OF FINAL CL/PL FILES TO ORIGAMI (QAT)
# NOTE: Keep main pipeline code unchanged; this cell only uploads final CSVs.
# =============================================================================

# If paramiko is not available in your environment, uncomment the next line
# to install it once for the workspace/session.
# %pip install paramiko

import os
import paramiko

# -----------------------------------------------------------------------------
# SFTP connection details (QAT)
# -----------------------------------------------------------------------------
SFTP_HOST = "live-ftp.origamirisk.com"
SFTP_PORT = 22
SFTP_USERNAME = "GMCTESFTP"

# IMPORTANT: For real use, store this in Azure Key Vault and retrieve via
# notebookutils.credentials.getSecret instead of hard-coding.
SFTP_PASSWORD = "9C*z8FH7*5uvmWN"  

# Remote target folders for final files
REMOTE_CL_DIR = "/AssureClaims/CP"  # CL file target directory
REMOTE_PL_DIR = "/AssureClaims/PA"  # PL file target directory

# -----------------------------------------------------------------------------
# Helper: copy Lakehouse CSV to local temp file
# -----------------------------------------------------------------------------

def copy_lakehouse_to_local(lh_path: str, local_dir: str = "/tmp") -> str:
    """Copy a file from Lakehouse Files (e.g. 'Files/CA/...csv') to a local path
    (e.g. '/tmp/CA_YYYYMMDD.csv') and return the local path.
    """
    os.makedirs(local_dir, exist_ok=True)
    filename = lh_path.split("/")[-1]
    local_path = os.path.join(local_dir, filename)

    # In Fabric Spark notebooks, prefix 'file:' for local FS paths
    notebookutils.fs.cp(lh_path, f"file:{local_path}", recurse=False)
    return local_path

# -----------------------------------------------------------------------------
# Helper: upload a single file to SFTP
# -----------------------------------------------------------------------------

def upload_to_sftp(local_path: str, remote_dir: str):
    """Upload local_path to SFTP into remote_dir, preserving the file name."""
    filename = os.path.basename(local_path)
    remote_path = f"{remote_dir.rstrip('/')}/{filename}"

    transport = paramiko.Transport((SFTP_HOST, SFTP_PORT))
    transport.connect(username=SFTP_USERNAME, password=SFTP_PASSWORD)

    sftp = paramiko.SFTPClient.from_transport(transport)
    try:
        # Ensure target directory exists (create if needed)
        try:
            sftp.chdir(remote_dir)
        except IOError:
            parts = remote_dir.strip("/").split("/")
            path_so_far = ""
            for part in parts:
                path_so_far += f"/{part}"
                try:
                    sftp.mkdir(path_so_far)
                except IOError:
                    # Directory already exists
                    pass

        print(f"Uploading {local_path} -> {remote_path} ...")
        sftp.put(local_path, remote_path)
        print(f"Upload complete: {remote_path}")
    finally:
        sftp.close()
        transport.close()

# -----------------------------------------------------------------------------
# MAIN: Upload CL and/or PL final files based on PARAM_LOB (not old LOB_SELECTION)
# Uses written_files dict created in the previous cell. Skips if file missing.
# -----------------------------------------------------------------------------

files_uploaded = []

if 'written_files' not in globals():
    raise RuntimeError("written_files dict not found. Please run the 'Write files to the lakehouse' cell first.")

# Upload CL only if a CL file was actually written
if lob_param in ["CL", "BOTH"] and "CL" in written_files:
    if notebookutils.fs.exists(written_files["CL"]):
        cl_local_path = copy_lakehouse_to_local(written_files["CL"])
        upload_to_sftp(cl_local_path, REMOTE_CL_DIR)
        files_uploaded.append(("CL", cl_local_path, REMOTE_CL_DIR))
    else:
        print("CL path in written_files does not exist -> skipping CL upload.")

# Upload PL only if a PL file was actually written
if lob_param in ["PL", "BOTH"] and "PL" in written_files:
    if notebookutils.fs.exists(written_files["PL"]):
        pl_local_path = copy_lakehouse_to_local(written_files["PL"])
        upload_to_sftp(pl_local_path, REMOTE_PL_DIR)
        files_uploaded.append(("PL", pl_local_path, REMOTE_PL_DIR))
    else:
        print("PL path in written_files does not exist -> skipping PL upload.")

print("\nSFTP upload summary:")
if not files_uploaded:
    print("  No files uploaded; either no data was available or PARAM_LOB excluded CL/PL.")
else:
    for lob, local_path, remote_dir in files_uploaded:
        print(f"  {lob}: {os.path.basename(local_path)} -> {remote_dir}")


StatementMeta(, 9c861cff-517b-43c1-b78c-c482bdfba29a, 31, Finished, Available, Finished, False)


SFTP upload summary:
  No files uploaded; either no data was available or PARAM_LOB excluded CL/PL.


In [30]:
import json

result = {
    "run_date": str(run_date),
    "load_type": load_type,
    "cl_rows": df_cl_final.count() if lob_param in ("CL", "BOTH") else 0,
    "pl_rows": df_pl_final.count() if lob_param in ("PL", "BOTH") else 0,
    "files_uploaded": len(files_uploaded)
}
result["has_data"] = result["files_uploaded"] > 0

notebookutils.notebook.exit(json.dumps(result))

StatementMeta(, 9c861cff-517b-43c1-b78c-c482bdfba29a, 32, Finished, Available, Finished, False)

ExitValue: {"run_date": "2026-11-01", "load_type": "INCREMENTAL", "cl_rows": 0, "pl_rows": 0, "files_uploaded": 0, "has_data": false}